# Figure 2: Equal-home diurnal profiles

Show typical local clock-hour profiles by season. Each home contributes one median per outcome-season-hour before the across-home median is calculated. Shaded bands show the interquartile range of the home medians across homes.

**Step:** 13. Article figure. See README.md for its data prerequisites.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Read the retained hourly CSV. Use local clock hours and preserve the two-stage aggregation so homes with more observations do not receive more weight.

**Outputs:** The named figure (PNG and PDF) in the separate reproduction figures directory.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure figure paths and resolution

Only presentation and output settings belong here. Keep the manuscript data and event specification fixed when reproducing the final figure.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Set a lower DPI only for an explicit draft; manuscript exports use 900.
FIGURE_DPI = int(os.environ.get("INDOOR_AQ_FIGURE_DPI", "900"))
os.environ["INDOOR_AQ_OUTPUT"] = str(output_root(OUTPUT))
os.environ["INDOOR_AQ_FIGURE_DPI"] = str(FIGURE_DPI)
# All event-result figures use this run's results folder.


## Load plotting libraries and define the figure settings

The following constants preserve the current figure's outcome order, labels, colors, panel geometry, and reference lines. Numerical methodology and interpretation are documented in the preceding and following cells.

In [ ]:
import os
import sys
from pathlib import Path
from shared import CODE, DATA, output_root, load_hourly as load_manuscript_hourly

import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, LogLocator, NullFormatter, ScalarFormatter

# ---- Input data ------------------------------------------------------------
DATA_DIR = DATA
HOURLY_PATH = DATA_DIR / 'hourly_primary_23.csv.gz'

# ---- Output ----------------------------------------------------------------
OUTPUT_DIR = output_root() / 'figures'
OUTPUT_STEM = 'Fig2_diurnal_profiles'
EXPORT_FORMATS = ('png', 'pdf')
DPI = int(os.environ.get('INDOOR_AQ_FIGURE_DPI', '900'))

# ---- Font and library versions used for the manuscript figure --------------
FONT_FAMILY = 'Liberation Sans'
FALLBACK_FONT = 'Arial'  # same character widths as Liberation Sans, so the layout is unchanged
# Folders searched for LiberationSans-*.ttf if the font is not installed.
FONT_SEARCH_DIRS = [
    Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts',
    Path(os.environ.get('LOCALAPPDATA', '')) / 'Microsoft' / 'Windows' / 'Fonts',
    Path(r'C:\Program Files\LibreOffice\share\fonts\truetype'),
    # Optional: a folder with LiberationSans-*.ttf (e.g. unzipped from the
    # Liberation fonts release) can be supplied through INDOOR_AQ_FONT_DIR.
    *([Path(os.environ['INDOOR_AQ_FONT_DIR'])] if os.environ.get('INDOOR_AQ_FONT_DIR') else []),
]
REQUIRED_PANDAS_MAJOR = 2     # portable CSV input, no pickle dependency
EXPECTED_MATPLOTLIB = '3.11'  # other releases may shift text by a sub-pixel

# ---- Panels (row by row: a b / c d) -----------------------------------------
VARIABLES = ['temp', 'rh', 'pm25', 'co2']
PANEL_LETTERS = 'abcd'
Y_LABELS = {
    'temp': 'Temperature (°C)',
    'rh': 'Relative humidity (%)',
    'pm25': r'PM$_{2.5}$ ($\mu$g/m$^3$)',
    'co2': r'CO$_2$ (ppm)',
}

# ---- Seasons ----------------------------------------------------------------
# Seasons are already assigned in the hourly data:
# non-heating = April–October, heating = November–March.
SEASONS = ['Non-heating season', 'Heating season']
SEASON_COLORS = {'Non-heating season': '#92C6A0', 'Heating season': '#DE9F83'}
SEASON_LINESTYLES = {'Non-heating season': '-', 'Heating season': '--'}
BAND_ALPHA = 0.28  # opacity of the interquartile-range bands

# ---- Axes ---------------------------------------------------------------------
HOUR_TICKS = [0, 4, 8, 12, 16, 20, 23]
PM25_TICKS = [7, 10, 15, 20, 30, 50, 70, 100, 150]   # labeled; unlabeled minor ticks in between
PM25_MARGIN = 1.15  # log-axis padding: lowest lower quartile / 1.15 and highest upper quartile * 1.15

# ---- Figure layout (inches and figure fractions) -----------------------------
FIGURE_SIZE = (13, 8.7)
SUBPLOT_LAYOUT = dict(left=0.085, right=0.985, bottom=0.13, top=0.96, wspace=0.22, hspace=0.25)
LEGEND_ANCHOR = (0.53, 0.015)

## Check environment

Check the data-library requirement and report a rendering-version difference. Reading CSV avoids the former pandas-specific pickle dependency. A version warning does not change the plotted observations.

In [ ]:
def check_environment():
    """Check library versions against those used for the manuscript figure.

    Raises
    ------
    ImportError
        If pandas is older than 2.0.

    Warns
    -----
    UserWarning
        If matplotlib is not 3.11.x. The plotted data are unaffected, but text
        can be placed a sub-pixel differently, so the PNG would not be
        pixel-identical to the manuscript version.
    """
    if int(pd.__version__.split('.')[0]) < REQUIRED_PANDAS_MAJOR:
        raise ImportError(
            f'pandas >= {REQUIRED_PANDAS_MAJOR}.0 is needed to read {HOURLY_PATH.name}; '
            f'this kernel has pandas {pd.__version__}.')
    if not matplotlib.__version__.startswith(EXPECTED_MATPLOTLIB + '.'):
        warnings.warn(
            f'The manuscript figure was rendered with matplotlib {EXPECTED_MATPLOTLIB}.x; '
            f'this kernel has {matplotlib.__version__}.')

## Apply style

Find the figure font (Liberation Sans, as in Figures 3, 4 and S1; Arial has the same character widths), then reset Matplotlib defaults before applying the figure's font, line, and export settings. This prevents unrelated notebook styling from carrying over into the manuscript figure.

In [ ]:
def resolve_font(family=FONT_FAMILY, search_dirs=FONT_SEARCH_DIRS, fallback=FALLBACK_FONT):
    """Return a font family that matplotlib can use, preferring ``family``.

    If ``family`` is not installed, any matching TrueType files found in
    ``search_dirs`` (e.g. LiberationSans-Regular.ttf) are registered for this
    session. If none are found, ``fallback`` is returned with a warning.
    """
    def available():
        return {font.name for font in font_manager.fontManager.ttflist}

    if family in available():
        return family

    pattern = family.replace(' ', '') + '-*.ttf'
    for directory in search_dirs:
        for path in sorted(Path(directory).glob(pattern)):
            font_manager.fontManager.addfont(str(path))

    if family in available():
        return family
    warnings.warn(f'{family} not found; using {fallback}. Glyph shapes will differ slightly.')
    return fallback


def apply_style(font_family):
    """Reset matplotlib to its defaults, then apply the Fig. 2 style.

    Resetting first keeps the output independent of earlier notebook cells
    and of any personal matplotlibrc file.
    """
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': font_family,
        'font.size': 12,
        'axes.labelsize': 14,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42,      # embed TrueType fonts so PDF text stays editable
        'svg.fonttype': 'none',  # keep SVG text as text rather than paths
    })

## Load hourly

Load hourly measurements through the shared sensor-based crosswalk. Missing observations stay missing; no additional filtering or concentration transformation is introduced here.

In [ ]:
def load_hourly(path=HOURLY_PATH):
    """Load the quality-checked hourly observations for the 23 homes.

    Returns
    -------
    pandas.DataFrame
        Long format, one row per home, variable and hour. Columns used here:
        ``home``, ``sensor``, ``variable`` ('temp', 'rh', 'pm25' or 'co2'),
        ``season``, ``hour`` (local clock hour, 0-23) and ``value``
        (NaN where the hour is missing or failed QA/QC).
    """
    return load_manuscript_hourly(os.environ.get('INDOOR_AQ_OUTPUT'), source_labels=True)

## Compute home hourly medians

Calculate each home's median from its available observations at each local clock hour within season and outcome. Keep the home-level aggregation explicit for equal weighting.

In [ ]:
def compute_home_hourly_medians(hourly):
    """Step 1: median of each home's valid values at each clock hour and season.

    Returns
    -------
    pandas.DataFrame
        One row per home, variable, season and hour, with the home's median
        (``home_median``) and the number of valid hours behind it (``n_valid``).
    """
    valid = hourly.dropna(subset=['value'])
    return (valid.groupby(['home', 'sensor', 'variable', 'season', 'hour'])['value']
            .agg(home_median='median', n_valid='size')
            .reset_index())

## Compute equal weight profiles

Take the median and the interquartile range (25th and 75th percentiles, linear interpolation) of the home-level clock-hour medians. The resulting curve describes the typical participating home and does not pool every raw hour into one weighted median.

In [ ]:
def compute_equal_weight_profiles(home_medians):
    """Step 2: median and interquartile range across homes at each clock hour.

    Each home counts once.

    Returns
    -------
    pandas.DataFrame
        One row per variable, season and hour, with the number of contributing
        homes (``n_homes``), the across-home ``median``, and the 25th and 75th
        percentiles of the home medians (``q25``, ``q75``; linear interpolation).
    """
    return (home_medians.groupby(['variable', 'season', 'hour'])['home_median']
            .agg(n_homes='count', median='median',
                 q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
            .reset_index())

## Draw profiles

Draw the two seasonal median profiles over bands showing the interquartile range across homes, with the fixed manuscript colors. Gaps reflect unavailable values rather than interpolated measurements.

In [ ]:
def draw_profiles(ax, profiles, variable):
    """Plot the non-heating and heating diurnal profiles of ``variable``.

    Each median curve is drawn over a band covering the interquartile range of
    the home medians across homes.
    """
    for season in SEASONS:
        profile = profiles[profiles['variable'].eq(variable)
                           & profiles['season'].eq(season)].sort_values('hour')
        ax.fill_between(profile['hour'], profile['q25'], profile['q75'],
                        color=SEASON_COLORS[season], alpha=BAND_ALPHA, lw=0, zorder=1)
        ax.plot(profile['hour'], profile['median'],
                color=SEASON_COLORS[season], ls=SEASON_LINESTYLES[season], lw=2.8)

## Format panel

Apply the panel's labels, tick formatting, limits, and panel letter. Axis display choices do not alter the model or measurement values.

In [ ]:
def format_panel(ax, variable, letter):
    """Set the hour axis, the y label, the grid and the panel letter."""
    ax.set_xticks(HOUR_TICKS)
    ax.set_xlim(0, 23)
    ax.set_xlabel('hour of day')
    ax.set_ylabel(Y_LABELS[variable])

    ax.grid(axis='y', color='#dddddd', lw=0.6, alpha=0.8)
    ax.set_axisbelow(True)

    # Panel letter just above the axes, left of the y axis
    ax.text(-0.11, 1.02, f'{letter})', transform=ax.transAxes,
            fontsize=17, fontweight='bold', va='bottom')

## Set log pm25 axis

Use logarithmic spacing while labeling ticks in the original concentration units. Labeled ticks run from 7 to 150 µg/m³ with unlabeled minor ticks between them, and the limits cover both seasons' bands. This display transform is separate from the log1p model sensitivity.

In [ ]:
def set_log_pm25_axis(ax, profiles):
    """Use a log y axis with plain-number labels, padded around both seasons' bands."""
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(FixedLocator(PM25_TICKS))
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10)))
    ax.yaxis.set_minor_formatter(NullFormatter())

    pm25 = profiles[profiles['variable'].eq('pm25')]
    ax.set_ylim(pm25['q25'].min() / PM25_MARGIN, pm25['q75'].max() * PM25_MARGIN)

## Add season legend

Use one shared seasonal legend to keep color interpretation consistent across panels.

In [ ]:
def add_season_legend(fig):
    """Add the shared season legend (band and line for each season) below the panels."""
    handles = [(Patch(facecolor=SEASON_COLORS[season], alpha=BAND_ALPHA, lw=0),
                Line2D([0], [0], color=SEASON_COLORS[season], ls=SEASON_LINESTYLES[season], lw=3))
               for season in SEASONS]
    fig.legend(handles=handles, labels=SEASONS, handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
               loc='lower center', ncol=2, frameon=False, bbox_to_anchor=LEGEND_ANCHOR,
               handlelength=3.2)

## Build figure

Assemble the complete manuscript panel layout from the validated inputs and drawing helpers. This function contains presentation logic and does not fit a statistical model.

In [ ]:
def build_figure(profiles):
    """Build the 2 x 2 panels of Fig. 2 and return the figure.

    Parameters
    ----------
    profiles : pandas.DataFrame
        Output of :func:`compute_equal_weight_profiles`.
    """
    fig, axes = plt.subplots(2, 2, figsize=FIGURE_SIZE)
    fig.subplots_adjust(**SUBPLOT_LAYOUT)

    for ax, variable, letter in zip(axes.flat, VARIABLES, PANEL_LETTERS):
        draw_profiles(ax, profiles, variable)
        format_panel(ax, variable, letter)
        if variable == 'pm25':
            set_log_pm25_axis(ax, profiles)

    add_season_legend(fig)
    return fig

## Save figure

Export the figure as PNG and PDF. PNG uses the configured resolution (900 dpi by default); PDF and SVG, when included, preserve vector elements. The output directory is separate from the manuscript source figures.

In [ ]:
def save_figure(fig, output_dir=OUTPUT_DIR, stem=OUTPUT_STEM,
                formats=EXPORT_FORMATS, dpi=DPI):
    """Save ``fig`` once per format, cropped to a tight bounding box.

    Returns the written paths.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in formats:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        paths.append(path)
    return paths

## Build and export the figure

Read the retained hourly CSV. Use local clock hours and preserve the two-stage aggregation so homes with more observations do not receive more weight. The cells above expose each processing and plotting step for inspection. The following execution writes the figure and prints its actual paths.

In [ ]:
check_environment()
apply_style(resolve_font())

hourly = load_hourly()
home_medians = compute_home_hourly_medians(hourly)
profiles = compute_equal_weight_profiles(home_medians)

fig = build_figure(profiles)
for path in save_figure(fig):
    print(f'Saved {path}')

## Display a compact notebook preview

The preview uses a smaller PNG rendered from the same figure object. The saved publication PNG remains at the configured 900 dpi. Keep methodological detail in the caption when preparing the manuscript.

In [ ]:
import io
from IPython.display import Image, display
_preview = io.BytesIO()
fig.savefig(_preview, format="png", dpi=110, bbox_inches="tight")
display(Image(data=_preview.getvalue()))
plt.close(fig)